Note: Due to GPU runtime and compute constraints, the 30-seed robustness experiments were executed in three batches. This notebook contains the consolidated 30-seed configuration used for the final robustness analysis.

## 1. Configuration and Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import pandas as pd
import numpy as np
import json
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import warnings
import time
from datetime import datetime
from urllib.parse import unquote
import random
from scipy import stats

# Suppress tokenizers parallelism warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings('ignore')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ============================================
# FIXED CONFIGURATION (from Phase 2 - Best Config)
# ============================================
LEARNING_RATE = 5e-5
BATCH_SIZE = 32
NUM_WORKERS = 2
EARLY_STOPPING_PATIENCE = 5
DROPOUT = 0.5
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 20


# ============================================
# ROBUSTNESS EXPERIMENT SETUP
# ============================================
SEED_RANGE_START = 1
SEED_RANGE_END = 500

SEEDS = [
    13, 14, 16, 17, 45, 48, 53, 58, 72, 102,
    112, 115, 120, 126, 141, 215, 217, 259, 280, 288,
    303, 309, 328, 333, 347, 360, 367, 378, 380, 457
]
NUM_SEEDS = len(SEEDS)
assert len(SEEDS) == 30, f"Expected 30 seeds, got {len(SEEDS)}: {SEEDS}"

# Fixed seed for model initialization (reproducibility)
MODEL_INIT_SEED = 42

# Data split ratios
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Output directories
RESULTS_DIR = "/content/drive/MyDrive/data255_revision_1/model/concat/result/real_30seeds_final"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "experiments"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "learning_curves"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "comparison_plots"), exist_ok=True)


print(f" Configuration loaded:")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Early Stopping Patience: {EARLY_STOPPING_PATIENCE}")
print(f"   Dropout: {DROPOUT}")
print(f"   Weight Decay: {WEIGHT_DECAY}")
print(f"   Max Epochs: {MAX_EPOCHS}")
print(f"\n Random Seeds Generated ({NUM_SEEDS} seeds from {SEED_RANGE_START}-{SEED_RANGE_END}):")
print(f"   Seeds: {SEEDS}")
print(f"\n Results directory: {RESULTS_DIR}")
print(f"   Model init seed: {MODEL_INIT_SEED} (fixed for all experiments)")

# Save seeds list for reference
with open(os.path.join(RESULTS_DIR, "seeds_list.txt"), 'w') as f:
    f.write("Generated Random Seeds for Robustness Experiments\n")
    f.write("=" * 50 + "\n")
    f.write(f"Total seeds: {NUM_SEEDS}\n")
    f.write(f"Range: {SEED_RANGE_START}-{SEED_RANGE_END}\n")
    f.write("=" * 50 + "\n\n")
    for i, seed in enumerate(SEEDS, 1):
        f.write(f"{i:2d}. Seed {seed}\n")
print(f"\n Seeds list saved to: {os.path.join(RESULTS_DIR, 'seeds_list.txt')}")


## 2. Load Full Dataset


In [ ]:
SPLITS_ROOT = "/content/drive/MyDrive/data255_revision_1/preprocessing/splits/phase3_30seeds"

def load_seed_csvs(seed):
    base = os.path.join(SPLITS_ROOT, f"seed_{seed}")
    train_df = pd.read_csv(os.path.join(base, "train.csv"))
    val_df = pd.read_csv(os.path.join(base, "val.csv"))
    test_df = pd.read_csv(os.path.join(base, "test.csv"))
    return train_df, val_df, test_df, base


# Use one seed only to define global metadata
_meta_train_df, _meta_val_df, _meta_test_df, _ = load_seed_csvs(SEEDS[0])
_meta_df = pd.concat([_meta_train_df, _meta_val_df, _meta_test_df], ignore_index=True)

all_styles = sorted(_meta_df['style'].str.strip().unique())
style_to_idx = {s: i for i, s in enumerate(all_styles)}
idx_to_style = {i: s for s, i in style_to_idx.items()}
num_classes = len(all_styles)

print(f"Styles: {num_classes} classes – {all_styles}")


## 3. Load Pre-trained Models


In [ ]:
!pip install git+https://github.com/openai/CLIP.git


In [ ]:
# Load CLIP model
import clip
print("Loading CLIP model...")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print("CLIP model loaded!")

# Load BERT-based text encoder
from transformers import AutoTokenizer, AutoModel
print("Loading BERT-based text encoder...")
fashionbert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
fashionbert_model = AutoModel.from_pretrained('bert-base-uncased').to(device)
fashionbert_model.eval()
print("BERT-based text encoder loaded!")


## 4. Dataset and Model Classes


In [ ]:
class FashionMultiModalDataset(Dataset):
    """Dataset class for multi-modal fashion style classification (same as Phase 2)"""

    def __init__(self, df, captions_dict, style_to_idx, transform=None, base_dir=None):
        self.df = df.reset_index(drop=True)
        self.captions_dict = captions_dict
        self.style_to_idx = style_to_idx
        self.transform = transform
        self.base_dir = base_dir

        # Filter out samples without captions or missing image files
        self.valid_indices = []
        self.missing_examples = []

        for idx in range(len(self.df)):
            row = self.df.iloc[idx]
            raw_image_path = row['image_path']

            # Resolve image path using current concat logic
            image_path = raw_image_path
            if base_dir and not os.path.isabs(image_path):
                image_path = os.path.join(base_dir, image_path)

            # Decode URL-encoded characters
            if '%' in image_path:
                path_parts = image_path.split('/')
                decoded_parts = [unquote(part) if '%' in part else part for part in path_parts]
                image_path = '/'.join(decoded_parts)

            has_caption = (raw_image_path in captions_dict) or (image_path in captions_dict)
            exists = os.path.exists(image_path)

            if has_caption and exists:
                self.valid_indices.append(idx)
            elif len(self.missing_examples) < 5:
                reason = []
                if not has_caption:
                    reason.append("missing_caption")
                if not exists:
                    reason.append("missing_image")
                self.missing_examples.append((raw_image_path, image_path, ",".join(reason)))

        print(f"Dataset initialized with {len(self.valid_indices)} valid samples (out of {len(self.df)})")

        if self.missing_examples:
            print("  Filtered examples:")
            for raw_path, resolved_path, reason in self.missing_examples:
                print(f"    reason={reason} raw={raw_path}")
                print(f"    resolved={resolved_path}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        row = self.df.iloc[actual_idx]

        # Load image
        image_path = row['image_path']

        # Convert to absolute path if needed
        if self.base_dir and not os.path.isabs(image_path):
            image_path = os.path.join(self.base_dir, image_path)

        # Decode URL-encoded characters in filename
        if '%' in image_path:
            path_parts = image_path.split('/')
            decoded_parts = [unquote(part) if '%' in part else part for part in path_parts]
            image_path = '/'.join(decoded_parts)

        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # Get caption
        caption = self.captions_dict.get(image_path,
                                        self.captions_dict.get(row['image_path'],
                                                              "No caption available"))

        # Get label
        style = row['style']
        label = self.style_to_idx[style]

        return {
            'image': image,
            'caption': caption,
            'label': label,
            'style': style,
            'image_path': image_path
        }

class ConcatMLPFusion(nn.Module):
    """
    Simple concat-then-MLP fusion:
    [image_emb | text_emb] → ReLU → Linear → ReLU → output (hidden_dim)
    """
    def __init__(self, visual_dim, textual_dim, hidden_dim=512):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(visual_dim + textual_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True)
        )

    def forward(self, visual_feat, textual_feat):
        # visual_feat: [B, visual_dim], textual_feat: [B, textual_dim]
        fused = torch.cat([visual_feat, textual_feat], dim=-1)
        return self.mlp(fused)
class MultiModalFashionClassifier(nn.Module):
    def __init__(self, clip_model, fashionbert_model, fashionbert_tokenizer,
                 num_classes, dropout=0.5, visual_dim=512, textual_dim=768):
        super().__init__()

        self.clip_model = clip_model
        self.fashionbert_model = fashionbert_model
        self.fashionbert_tokenizer = fashionbert_tokenizer

        # freeze encoders
        for p in self.clip_model.parameters():
            p.requires_grad = False
        for p in self.fashionbert_model.parameters():
            p.requires_grad = False

        self.fusion = ConcatMLPFusion(visual_dim, textual_dim, hidden_dim=512)

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

        self.clip_model.eval()
        self.fashionbert_model.eval()

    def train(self, mode=True):
        super().train(mode)
        self.clip_model.eval()
        self.fashionbert_model.eval()
        return self

    def forward(self, images, captions):
        with torch.no_grad():
            visual_features = self.clip_model.encode_image(images).float()

            inputs = self.fashionbert_tokenizer(
                list(captions),
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128,
            ).to(images.device)

            outputs = self.fashionbert_model(**inputs)
            textual_features = outputs.last_hidden_state[:, 0, :]

        fused_features = self.fusion(visual_features, textual_features)
        return self.classifier(fused_features)

print(" Model classes defined!")


In [ ]:
def make_seed_worker(base_seed):
    def seed_worker(worker_id):
        worker_seed = base_seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)
    return seed_worker


In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(train_loader, desc="Training", leave=False):
        images = batch['image'].to(device)
        captions = batch['caption']
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(images, captions)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = correct / total if total > 0 else 0.0
    return total_loss / len(train_loader), accuracy

def validate_epoch(model, val_loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation", leave=False):
            images = batch['image'].to(device)
            captions = batch['caption']
            labels = batch['label'].to(device)

            logits = model(images, captions)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate macro F1
    val_macro_f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0) if len(all_predictions) > 0 else 0.0

    accuracy = correct / total if total > 0 else 0.0
    return total_loss / len(val_loader), accuracy, all_predictions, all_labels, val_macro_f1

print(" Training functions defined!")


## 6. Experiment Runner Function


In [ ]:
def run_robustness_experiment(seed_value, seed_idx, style_to_idx,
                              clip_model, clip_preprocess,
                              fashionbert_model, fashionbert_tokenizer,
                              num_classes, device, all_styles, base_dir):
    """
    Run a single robustness experiment with given seed

    Args:
        seed_value: Random seed value for data splitting (from 1-500)
        seed_idx: Index of seed in SEEDS list (1-30)
        df_full: Full dataset DataFrame
        captions_dict: Dictionary mapping image paths to captions
        style_to_idx: Dictionary mapping style names to indices
        clip_model, fashionbert_model: Pre-trained models
        fashionbert_tokenizer: BERT tokenizer
        num_classes: Number of classes
        device: Device
        all_styles: List of style names
        base_dir: Base directory for absolute paths

    Returns:
        Dictionary with all results and metrics
    """


    base = os.path.join(SPLITS_ROOT, f"seed_{seed_value}")
    train_df = pd.read_csv(os.path.join(base, "train.csv"))
    val_df = pd.read_csv(os.path.join(base, "val.csv"))
    test_df = pd.read_csv(os.path.join(base, "test.csv"))

    train_captions = dict(zip(train_df["image_path"], train_df["caption"]))
    val_captions = dict(zip(val_df["image_path"], val_df["caption"]))
    test_captions = dict(zip(test_df["image_path"], test_df["caption"]))


    print(f"\n{'='*70}")
    print(f"Experiment {seed_idx}/{len(SEEDS)}: Seed {seed_value}")
    print(f"{'='*70}")

    # Check if result already exists (resume capability)
    result_file = os.path.join(RESULTS_DIR, "experiments", f"seed_{seed_value}_results.json")
    if os.path.exists(result_file):
        print(f"    Result already exists, skipping...")
        with open(result_file, 'r') as f:
            return json.load(f)

    # Set fixed seed for model initialization (same for all experiments)
    torch.manual_seed(MODEL_INIT_SEED)
    np.random.seed(MODEL_INIT_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(MODEL_INIT_SEED)


    # Create datasets


    train_dataset = FashionMultiModalDataset(
        train_df, train_captions, style_to_idx,
        transform=clip_preprocess,
        base_dir=base_dir
    )

    val_dataset = FashionMultiModalDataset(
        val_df, val_captions, style_to_idx,
        transform=clip_preprocess,
        base_dir=base_dir
    )

    test_dataset = FashionMultiModalDataset(
        test_df, test_captions, style_to_idx,
        transform=clip_preprocess,
        base_dir=base_dir
    )
    # Create data loaders
    loader_seed = MODEL_INIT_SEED + seed_value
    g = torch.Generator()
    g.manual_seed(loader_seed)
    worker_init_fn = make_seed_worker(loader_seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        worker_init_fn=worker_init_fn,
        generator=g,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        worker_init_fn=worker_init_fn,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        worker_init_fn=worker_init_fn,
    )

    # Compute class weights using the actual filtered training samples
    train_valid_df = train_dataset.df.iloc[train_dataset.valid_indices]

    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.array(list(style_to_idx.values())),
        y=train_valid_df['style'].map(style_to_idx).values
    )
    class_weights = torch.FloatTensor(class_weights).to(device)

    # Initialize model (with fixed seed)
    model = MultiModalFashionClassifier(
        clip_model=clip_model,
        fashionbert_model=fashionbert_model,
        fashionbert_tokenizer=fashionbert_tokenizer,
        num_classes=num_classes,
        dropout=DROPOUT
    ).to(device)

    # Setup training
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(
        list(model.fusion.parameters()) + list(model.classifier.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    # Training tracking
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    val_macro_f1s = []
    learning_rates = []

    best_val_macro_f1 = 0
    best_val_loss = float('inf')
    best_epoch = 0
    patience_counter = 0
    early_stopped = False

    start_time = time.time()

    # Training loop with early stopping
    for epoch in range(MAX_EPOCHS):
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)

        # Validate
        val_loss, val_acc, val_preds, val_labels, val_macro_f1 = validate_epoch(
            model, val_loader, criterion, device
        )

        # Update scheduler
        scheduler.step()
        learning_rates.append(scheduler.get_last_lr()[0])

        # Store metrics
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        val_macro_f1s.append(val_macro_f1)

        # Track best Macro F1 (for saving & early stopping)
        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_epoch = epoch + 1
            patience_counter = 0
            # Save best model
            torch.save(model.state_dict(),
                      os.path.join(RESULTS_DIR, "models", f"seed_{seed_value}_best_model.pth"))
        else:
            patience_counter += 1

        # Track best loss (for overfitting detection)
        if val_loss < best_val_loss:
            best_val_loss = val_loss

        # Early stopping (based on Macro F1)
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            early_stopped = True
            print(f"  Early stopping at epoch {epoch+1} (no improvement for {EARLY_STOPPING_PATIENCE} epochs)")
            break

        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1}/{MAX_EPOCHS}: "
                  f"Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, "
                  f"Val Macro F1={val_macro_f1:.4f}")

    total_time = time.time() - start_time

    # Load best model for final evaluation
    model.load_state_dict(torch.load(
        os.path.join(RESULTS_DIR, "models", f"seed_{seed_value}_best_model.pth")))
    model.eval()

    # Final evaluation on test set
    test_loss, test_acc, test_preds, test_labels, test_macro_f1 = validate_epoch(
        model, test_loader, criterion, device
    )

    # Detect overfitting
    if len(val_losses) > best_epoch:
        val_loss_after_best = min(val_losses[best_epoch:])
        overfitting_detected = val_loss_after_best > best_val_loss * 1.05
    else:
        overfitting_detected = False

    # Calculate train/val gap at best epoch
    train_val_gap = train_losses[best_epoch - 1] - best_val_loss if best_epoch > 0 else 0

    # Create learning curves plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Loss curves
    axes[0, 0].plot(train_losses, label='Train Loss', color='blue', linewidth=2)
    axes[0, 0].plot(val_losses, label='Val Loss', color='red', linewidth=2)
    axes[0, 0].axvline(x=best_epoch-1, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch {best_epoch}')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Accuracy curves
    axes[0, 1].plot(train_accs, label='Train Acc', color='blue', linewidth=2)
    axes[0, 1].plot(val_accs, label='Val Acc', color='red', linewidth=2)
    axes[0, 1].axvline(x=best_epoch-1, color='green', linestyle='--', alpha=0.7)
    axes[0, 1].set_title('Training and Validation Accuracy')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Macro F1 curve
    axes[1, 0].plot(val_macro_f1s, label='Val Macro F1', color='green', linewidth=2)
    axes[1, 0].axvline(x=best_epoch-1, color='red', linestyle='--', alpha=0.7)
    axes[1, 0].axhline(y=best_val_macro_f1, color='red', linestyle='--', alpha=0.7,
                       label=f'Best: {best_val_macro_f1:.4f}')
    axes[1, 0].set_title('Validation Macro F1-Score')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Macro F1')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Summary text
    axes[1, 1].axis('off')
    summary_text = f"""
Seed: {seed_value} (Experiment {seed_idx}/{len(SEEDS)})
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Best Epoch: {best_epoch}
Early Stopped: {early_stopped}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Best Val Macro F1: {best_val_macro_f1:.4f}
Test Macro F1: {test_macro_f1:.4f}
Test Accuracy: {test_acc:.4f}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Overfitting: {'Yes' if overfitting_detected else 'No'}
Training Time: {total_time/60:.1f} minutes
    """
    axes[1, 1].text(0.1, 0.5, summary_text, fontsize=10, family='monospace',
                    verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.suptitle(f'Learning Curves: Seed {seed_value} (Experiment {seed_idx}/{len(SEEDS)})',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()

    # Save plot
    plot_path = os.path.join(RESULTS_DIR, "learning_curves", f"seed_{seed_value}_learning_curves.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()

    # Prepare results dictionary
    results = {
        "experiment_id": f"seed_{seed_value}",
        "seed_value": seed_value,
        "seed_index": seed_idx,
        "timestamp": datetime.now().isoformat(),
        "configuration": {
            "learning_rate": float(LEARNING_RATE),
            "batch_size": BATCH_SIZE,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "dropout": DROPOUT,
            "weight_decay": float(WEIGHT_DECAY),
            "max_epochs": MAX_EPOCHS,
            "model_init_seed": MODEL_INIT_SEED,
            "data_split_seed": seed_value,
            "dataset_size": "100% (full dataset)",
            "fusion": "concat_mlp",
            "image_preprocess": "clip_preprocess",
            "encoder_trainable": False,
            "encoder_mode": "eval",
            "optimizer_scope": "fusion+classifier",
            "scheduler": "CosineAnnealingLR",
            "scheduler_T_max": MAX_EPOCHS,
            "accuracy_scale": "0_to_1",
            "loader_seed": loader_seed,
        },
        "training_info": {
            "total_epochs": len(train_losses),
            "best_epoch": best_epoch,
            "early_stopped": early_stopped,
            "total_time_minutes": float(total_time / 60)
        },
        "validation_metrics": {
            "best_val_macro_f1": float(best_val_macro_f1),
            "best_val_accuracy": float(val_accs[best_epoch - 1]) if best_epoch > 0 else 0.0,
            "best_val_loss": float(best_val_loss),
            "final_val_macro_f1": float(val_macro_f1s[-1]),
            "final_val_accuracy": float(val_accs[-1]),
            "final_val_loss": float(val_losses[-1])
        },
        "test_metrics": {
            "test_macro_f1": float(test_macro_f1),
            "test_accuracy": float(test_acc),
            "test_loss": float(test_loss)
        },
        "overfitting_analysis": {
            "overfitting_detected": overfitting_detected,
            "best_val_loss": float(best_val_loss),
            "val_loss_after_best": float(val_losses[best_epoch:][0]) if len(val_losses) > best_epoch else float(val_losses[-1]),
            "increase_percentage": float((val_losses[best_epoch:][0] - best_val_loss) / best_val_loss * 100) if len(val_losses) > best_epoch else 0.0,
            "train_val_gap": float(train_val_gap)
        },
        "training_curves": {
            "train_losses": [float(x) for x in train_losses],
            "val_losses": [float(x) for x in val_losses],
            "train_accs": [float(x) for x in train_accs],
            "val_accs": [float(x) for x in val_accs],
            "val_macro_f1s": [float(x) for x in val_macro_f1s],
            "learning_rates": [float(x) for x in learning_rates]
        },
        "data_split_info": {
            "train_size": len(train_df),
            "val_size": len(val_df),
            "test_size": len(test_df)
        }
    }

    # Save results JSON
    json_path = os.path.join(RESULTS_DIR, "experiments", f"seed_{seed_value}_results.json")
    with open(json_path, 'w') as f:
        json.dump(results, f, indent=2)

    print(f"   Completed: Best Val Macro F1={best_val_macro_f1:.4f}, "
          f"Test Macro F1={test_macro_f1:.4f}, "
          f"Overfitting={'Yes' if overfitting_detected else 'No'}")

    return results

print(" Experiment runner function defined!")


In [ ]:
import time

# Get base directory for absolute paths
BASE_DIR = "/content/drive/Shareddrives/data255/data255"

# Run all robustness experiments
overall_start = time.time()

all_results = []
failed_seeds = []

print(f"\n{'='*70}")
print(f"STARTING ROBUSTNESS EXPERIMENTS")
print(f"Total experiments: {len(SEEDS)}")
print("Dataset: pre-split CSVs")
print(f"Estimated time: ~{len(SEEDS) * 2.5:.1f} hours")
print(f"{'='*70}")

for seed_idx, seed_value in enumerate(SEEDS, 1):
    seed_start = time.time()

    try:
        result = run_robustness_experiment(
            seed_value=seed_value,
            seed_idx=seed_idx,
            style_to_idx=style_to_idx,
            clip_model=clip_model,
            clip_preprocess=clip_preprocess,
            fashionbert_model=fashionbert_model,
            fashionbert_tokenizer=fashionbert_tokenizer,
            num_classes=num_classes,
            device=device,
            all_styles=all_styles,
            base_dir=BASE_DIR
        )

        all_results.append(result)

        seed_minutes = (time.time() - seed_start) / 60

        # Progress update
        remaining = len(SEEDS) - seed_idx
        print(f"\nSeed {seed_value} finished or skipped in {seed_minutes:.2f} minutes")
        print(f"\n Progress: {seed_idx}/{len(SEEDS)} completed, {remaining} remaining")

    except Exception as e:
        seed_minutes = (time.time() - seed_start) / 60
        print(f"\n Error in seed {seed_value} after {seed_minutes:.2f} minutes: {e}")
        failed_seeds.append((seed_value, str(e)))
        import traceback
        traceback.print_exc()
        continue

overall_minutes = (time.time() - overall_start) / 60

print(f"\n{'='*70}")
print(f"ALL EXPERIMENTS COMPLETED!")
print(f"  Successful: {len(all_results)}/{len(SEEDS)}")
if failed_seeds:
    print(f"  Failed: {len(failed_seeds)} seeds")
    print(f"  Failed seeds: {[s[0] for s in failed_seeds]}")
print(f"  Total runtime: {overall_minutes:.2f} minutes")
print(f"{'='*70}")

# Save summary of completed experiments
summary = {
    "total_seeds": len(SEEDS),
    "successful_experiments": len(all_results),
    "failed_experiments": len(failed_seeds),
    "failed_seeds": [{"seed": s[0], "error": s[1]} for s in failed_seeds],
    "completed_seeds": [r["seed_value"] for r in all_results],
    "total_runtime_minutes": overall_minutes
}

with open(os.path.join(RESULTS_DIR, "experiments_summary.json"), 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n Experiments summary saved to: {os.path.join(RESULTS_DIR, 'experiments_summary.json')}")


## 8. Statistical Analysis


In [ ]:
# Load all results if not already loaded
if len(all_results) == 0:
    print("Loading results from JSON files...")
    all_results = []
    for seed_value in SEEDS:
        result_file = os.path.join(RESULTS_DIR, "experiments", f"seed_{seed_value}_results.json")
        if os.path.exists(result_file):
            with open(result_file, 'r') as f:
                all_results.append(json.load(f))
    print(f"Loaded {len(all_results)} results")

if len(all_results) == 0:
    print("  No results found! Please run experiments first.")
else:
    # Extract key metrics
    test_f1s = [r['test_metrics']['test_macro_f1'] for r in all_results]
    test_accs = [r['test_metrics']['test_accuracy'] for r in all_results]
    best_val_f1s = [r['validation_metrics']['best_val_macro_f1'] for r in all_results]
    train_val_gaps = [r['overfitting_analysis']['train_val_gap'] for r in all_results]
    overfitting_flags = [r['overfitting_analysis']['overfitting_detected'] for r in all_results]
    training_times = [r['training_info']['total_time_minutes'] for r in all_results]

    # Calculate statistics
    def calculate_stats(values, name):
        mean_val = np.mean(values)
        std_val = np.std(values)
        min_val = np.min(values)
        max_val = np.max(values)
        median_val = np.median(values)
        q25 = np.percentile(values, 25)
        q75 = np.percentile(values, 75)
        cv = (std_val / mean_val * 100) if mean_val != 0 else 0  # Coefficient of Variation

        # 95% Confidence Interval
        if len(values) > 1:
            ci = stats.t.interval(0.95, len(values)-1, loc=mean_val, scale=stats.sem(values))
            ci_lower, ci_upper = ci
        else:
            ci_lower, ci_upper = mean_val, mean_val

        return {
            "metric": name,
            "mean": float(mean_val),
            "std": float(std_val),
            "min": float(min_val),
            "max": float(max_val),
            "median": float(median_val),
            "q25": float(q25),
            "q75": float(q75),
            "cv_percent": float(cv),
            "ci_95_lower": float(ci_lower),
            "ci_95_upper": float(ci_upper),
            "n": len(values)
        }

    # Create statistics table
    stats_data = [
        calculate_stats(test_f1s, "Test Macro F1"),
        calculate_stats(test_accs, "Test Accuracy"),
        calculate_stats(best_val_f1s, "Best Val Macro F1"),
        calculate_stats(train_val_gaps, "Train/Val Gap"),
        calculate_stats(training_times, "Training Time (min)")
    ]

    df_stats = pd.DataFrame(stats_data)

    print("\n" + "="*80)
    print("STATISTICAL SUMMARY - Robustness Experiments")
    print("="*80)
    print(df_stats.to_string(index=False))

    # Save statistics
    stats_path = os.path.join(RESULTS_DIR, "statistical_analysis.json")
    with open(stats_path, 'w') as f:
        json.dump({
            "summary_statistics": stats_data,
            "overfitting_count": sum(overfitting_flags),
            "overfitting_percentage": sum(overfitting_flags) / len(overfitting_flags) * 100,
            "total_experiments": len(all_results)
        }, f, indent=2)

    df_stats.to_csv(os.path.join(RESULTS_DIR, "summary_table.csv"), index=False)

    print(f"\n Statistical analysis saved to:")
    print(f"   - {stats_path}")
    print(f"   - {os.path.join(RESULTS_DIR, 'summary_table.csv')}")

    # Print key insights
    print(f"\n{'='*80}")
    print("KEY INSIGHTS")
    print("="*80)
    print(f"Test Macro F1: {df_stats[df_stats['metric']=='Test Macro F1']['mean'].values[0]:.4f} ± {df_stats[df_stats['metric']=='Test Macro F1']['std'].values[0]:.4f}")
    print(f"   Range: [{df_stats[df_stats['metric']=='Test Macro F1']['min'].values[0]:.4f}, {df_stats[df_stats['metric']=='Test Macro F1']['max'].values[0]:.4f}]")
    print(f"   95% CI: [{df_stats[df_stats['metric']=='Test Macro F1']['ci_95_lower'].values[0]:.4f}, {df_stats[df_stats['metric']=='Test Macro F1']['ci_95_upper'].values[0]:.4f}]")
    print(f"   CV: {df_stats[df_stats['metric']=='Test Macro F1']['cv_percent'].values[0]:.2f}%")
    print(f"\nOverfitting: {sum(overfitting_flags)}/{len(overfitting_flags)} experiments ({sum(overfitting_flags)/len(overfitting_flags)*100:.1f}%)")
    print(f"Average training time: {np.mean(training_times):.1f} minutes per experiment")


## 9. Visualization and Comparison


In [ ]:
if len(all_results) > 0:
    # Extract metrics
    test_f1s = [r['test_metrics']['test_macro_f1'] for r in all_results]
    test_accs = [r['test_metrics']['test_accuracy'] for r in all_results]
    best_val_f1s = [r['validation_metrics']['best_val_macro_f1'] for r in all_results]
    train_val_gaps = [r['overfitting_analysis']['train_val_gap'] for r in all_results]
    overfitting_flags = [r['overfitting_analysis']['overfitting_detected'] for r in all_results]
    seed_values = [r['seed_value'] for r in all_results]

    # Create comprehensive comparison plot
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # Plot 1: Test Macro F1 distribution (Box plot)
    axes[0, 0].boxplot(test_f1s, vert=True, patch_artist=True,
                       boxprops=dict(facecolor='lightblue', alpha=0.7),
                       medianprops=dict(color='red', linewidth=2))
    axes[0, 0].scatter([1] * len(test_f1s), test_f1s, alpha=0.3, s=20, color='gray')
    axes[0, 0].axhline(y=np.mean(test_f1s), color='green', linestyle='--',
                      label=f'Mean: {np.mean(test_f1s):.4f}')
    axes[0, 0].set_ylabel('Test Macro F1-Score')
    axes[0, 0].set_title('Test Macro F1 Distribution', fontsize=12, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    axes[0, 0].legend()

    # Plot 2: Test Macro F1 vs Seed (Scatter)
    colors = ['red' if of else 'green' for of in overfitting_flags]
    axes[0, 1].scatter(seed_values, test_f1s, c=colors, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    axes[0, 1].axhline(y=np.mean(test_f1s), color='blue', linestyle='--', alpha=0.7, label='Mean')
    axes[0, 1].set_xlabel('Seed Value')
    axes[0, 1].set_ylabel('Test Macro F1-Score')
    axes[0, 1].set_title('Test Macro F1 vs Seed Value', fontsize=12, fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Plot 3: Histogram of Test Macro F1
    axes[0, 2].hist(test_f1s, bins=15, color='skyblue', edgecolor='black', alpha=0.7)
    axes[0, 2].axvline(x=np.mean(test_f1s), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(test_f1s):.4f}')
    axes[0, 2].axvline(x=np.median(test_f1s), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(test_f1s):.4f}')
    axes[0, 2].set_xlabel('Test Macro F1-Score')
    axes[0, 2].set_ylabel('Frequency')
    axes[0, 2].set_title('Test Macro F1 Distribution (Histogram)', fontsize=12, fontweight='bold')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3, axis='y')

    # Plot 4: Test Accuracy distribution
    axes[1, 0].boxplot(test_accs, vert=True, patch_artist=True,
                       boxprops=dict(facecolor='lightcoral', alpha=0.7),
                       medianprops=dict(color='red', linewidth=2))
    axes[1, 0].scatter([1] * len(test_accs), test_accs, alpha=0.3, s=20, color='gray')
    axes[1, 0].axhline(y=np.mean(test_accs), color='green', linestyle='--',
                      label=f'Mean: {np.mean(test_accs):.4f}')
    axes[1, 0].set_ylabel('Test Accuracy')
    axes[1, 0].set_title('Test Accuracy Distribution', fontsize=12, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    axes[1, 0].legend()

    # Plot 5: Train/Val Gap distribution
    axes[1, 1].boxplot(train_val_gaps, vert=True, patch_artist=True,
                       boxprops=dict(facecolor='lightyellow', alpha=0.7),
                       medianprops=dict(color='red', linewidth=2))
    axes[1, 1].scatter([1] * len(train_val_gaps), train_val_gaps, alpha=0.3, s=20, color='gray')
    axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    axes[1, 1].axhline(y=np.mean(train_val_gaps), color='green', linestyle='--',
                      label=f'Mean: {np.mean(train_val_gaps):.4f}')
    axes[1, 1].set_ylabel('Train-Val Loss Gap')
    axes[1, 1].set_title('Train-Val Gap Distribution', fontsize=12, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    axes[1, 1].legend()

    # Plot 6: Overfitting summary
    of_counts = [sum(overfitting_flags), len(overfitting_flags) - sum(overfitting_flags)]
    axes[1, 2].pie(of_counts, labels=['Overfitting', 'No Overfitting'], autopct='%1.1f%%',
                   colors=['red', 'green'], startangle=90, textprops={'fontsize': 10})
    axes[1, 2].set_title('Overfitting Summary', fontsize=12, fontweight='bold')

    plt.suptitle('Robustness Experiments - Statistical Analysis', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()

    # Save comparison plot
    comparison_path = os.path.join(RESULTS_DIR, "comparison_plots", "robustness_analysis.png")
    plt.savefig(comparison_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f" Comparison plots saved to: {comparison_path}")

    # Create learning curves overlay (sample of 5 experiments)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Sample 5 experiments for overlay
    sample_indices = np.linspace(0, len(all_results)-1, min(5, len(all_results)), dtype=int)

    # Plot validation loss curves
    for idx in sample_indices:
        result = all_results[idx]
        axes[0].plot(result['training_curves']['val_losses'],
                    alpha=0.6, linewidth=1.5, label=f"Seed {result['seed_value']}")
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Validation Loss')
    axes[0].set_title('Validation Loss Curves (Sample)', fontsize=12, fontweight='bold')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    # Plot validation Macro F1 curves
    for idx in sample_indices:
        result = all_results[idx]
        axes[1].plot(result['training_curves']['val_macro_f1s'],
                    alpha=0.6, linewidth=1.5, label=f"Seed {result['seed_value']}")
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Validation Macro F1')
    axes[1].set_title('Validation Macro F1 Curves (Sample)', fontsize=12, fontweight='bold')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('Learning Curves Overlay (Sample of 5 Experiments)', fontsize=14, fontweight='bold')
    plt.tight_layout()

    curves_path = os.path.join(RESULTS_DIR, "comparison_plots", "learning_curves_overlay.png")
    plt.savefig(curves_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f" Learning curves overlay saved to: {curves_path}")
else:
    print("  No results available for visualization")


## 10. Final Summary


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd
import json
import os
from sklearn.metrics import classification_report
import torchvision.transforms as transforms

print("Starting Patch: Generating missing reports for existing seeds...")

# 1. Define missing transformation variables
PATCH_TRANSFORM = clip_preprocess

for seed_value in SEEDS:
    json_path = os.path.join(RESULTS_DIR, "experiments", f"seed_{seed_value}_results.json")
    model_path = os.path.join(RESULTS_DIR, "models", f"seed_{seed_value}_best_model.pth")

    if os.path.exists(json_path) and os.path.exists(model_path):
        with open(json_path, 'r') as f:
            res = json.load(f)

        # Check if the patch has already been applied to avoid redundant processing
        if 'report' in res['test_metrics']:
            print(f"Seed {seed_value} already has report. Skipping.")
            continue

        print(f"Patching Seed {seed_value}...")

        # 2. Prepare data for the current Seed
        base_split_path = os.path.join(SPLITS_ROOT, f"seed_{seed_value}")
        test_df = pd.read_csv(f'{base_split_path}/test.csv')

        test_captions = dict(zip(test_df["image_path"], test_df["caption"]))

        # Use PATCH_TRANSFORM to replace the missing transform in the dataset
        test_dataset = FashionMultiModalDataset(
            test_df,
            test_captions,
            style_to_idx,
            PATCH_TRANSFORM,
            base_dir=BASE_DIR
        )
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

        # 3. Load model (verify num_classes is correct for the specific architecture)
        model = MultiModalFashionClassifier(
            clip_model=clip_model,
            fashionbert_model=fashionbert_model,
            fashionbert_tokenizer=fashionbert_tokenizer,
            num_classes=num_classes,
            dropout=DROPOUT
        ).to(device)
        model.load_state_dict(torch.load(model_path))
        model.eval()

        # 4. Inference and Report Generation
        # Note: This calls the previously defined validate_epoch function
        _, _, test_preds, test_labels, _ = validate_epoch(model, test_loader, nn.CrossEntropyLoss(), device)

        # Generate classification report using the defined style categories
        report = classification_report(test_labels, test_preds, target_names=all_styles, output_dict=True)

        # 5. Update the result dictionary and overwrite the saved JSON file
        res['test_metrics']['report'] = report
        with open(json_path, 'w') as f:
            json.dump(res, f, indent=2)

print(" All existing JSON files have been patched with 'report'!")


In [ ]:
# Reload the data from patched JSON files
all_results = []
for seed_value in SEEDS:
    res_path = os.path.join(RESULTS_DIR, "experiments", f"seed_{seed_value}_results.json")
    if os.path.exists(res_path):
        with open(res_path, 'r') as f:
            all_results.append(json.load(f))


In [ ]:
# ============================================
# Section 9: Aggregated Per-Class Analysis (Updated Structure)
# ============================================
if len(all_results) > 0:
    per_class_stats = {}

    # 1. Check data structure and find a valid sample for initialization
    valid_sample = None
    for r in all_results:
        # Metrics are located inside 'test_metrics' -> 'report' based on the patch structure
        if 'test_metrics' in r and 'report' in r['test_metrics']:
            valid_sample = r
            break

    if valid_sample is None:
        print(" Error: Could not find 'test_metrics' -> 'report' in all_results!")
        print("Current data keys:", all_results[0].keys())
    else:
        # Fetching f1, precision, and recall from the nested report structure
        report_data = valid_sample['test_metrics']['report']
        all_styles = sorted([k for k in report_data.keys()
                            if k not in ['accuracy', 'macro avg', 'weighted avg']])

        print(f"Detected {len(all_styles)} styles: {all_styles}")

        for style in all_styles:
            test_f1s_style = []
            test_precisions_style = []
            test_recalls_style = []

            for r in all_results:
                if 'test_metrics' in r and 'report' in r['test_metrics']:
                    curr_report = r['test_metrics']['report']
                    if style in curr_report:
                        test_f1s_style.append(curr_report[style]['f1-score'])
                        test_precisions_style.append(curr_report[style]['precision'])
                        test_recalls_style.append(curr_report[style]['recall'])

            if test_f1s_style:
                per_class_stats[style] = {
                    'test': {
                        'f1': calculate_stats(test_f1s_style, f"{style} F1"),
                        'precision': calculate_stats(test_precisions_style, f"{style} Precision"),
                        'recall': calculate_stats(test_recalls_style, f"{style} Recall")
                    }
                }

        # 2. Generate summary CSV table for reporting
        per_class_summary = []
        for style in per_class_stats.keys():
            s = per_class_stats[style]['test']['f1']
            per_class_summary.append({
                'Style': style,
                'F1_Mean': s['mean'],
                'F1_Std': s['std'],
                'F1_CV%': s['cv_percent'],
                'Precision_Mean': per_class_stats[style]['test']['precision']['mean'],
                'Recall_Mean': per_class_stats[style]['test']['recall']['mean']
            })

        df_per_class = pd.DataFrame(per_class_summary)
        df_per_class.to_csv(os.path.join(RESULTS_DIR, "per_class_metrics_summary.csv"), index=False)

        print("\n" + "="*50)
        print(f"PER-CLASS PERFORMANCE SUMMARY ({len(all_results)} SEEDS)")
        print("="*50)
        print(df_per_class[['Style', 'F1_Mean', 'F1_Std', 'F1_CV%']].to_string(index=False))


In [ ]:
# ============================================
# Section 10: Per-Class Visualizations
# ============================================
if len(all_results) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    styles_list = all_styles
    x_pos = np.arange(len(styles_list))

    # 1. F1-Score Bar Chart with Error Bars
    f1_means = [per_class_stats[s]['test']['f1']['mean'] for s in styles_list]
    f1_stds = [per_class_stats[s]['test']['f1']['std'] for s in styles_list]
    axes[0, 0].bar(x_pos, f1_means, yerr=f1_stds, capsize=5, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 0].set_title('Per-Class F1-Score (Mean ± Std)', fontweight='bold')
    axes[0, 0].set_xticks(x_pos)
    axes[0, 0].set_xticklabels(styles_list, rotation=45, ha='right')
    axes[0, 0].axhline(y=np.mean(f1_means), color='red', linestyle='--', label=f'Mean: {np.mean(f1_means):.4f}')
    axes[0, 0].legend()

    # 2. Precision vs Recall Scatter
    p_means = [per_class_stats[s]['test']['precision']['mean'] for s in styles_list]
    r_means = [per_class_stats[s]['test']['recall']['mean'] for s in styles_list]
    axes[0, 1].scatter(p_means, r_means, s=100, alpha=0.6, edgecolors='black')
    for i, txt in enumerate(styles_list):
        axes[0, 1].annotate(txt, (p_means[i], r_means[i]), fontsize=8)
    axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=0.2)
    axes[0, 1].set_title('Precision vs Recall (Mean)', fontweight='bold')
    axes[0, 1].set_xlabel('Precision')
    axes[0, 1].set_ylabel('Recall')

    # 3. Stability Analysis (CV%)
    cv_values = [per_class_stats[s]['test']['f1']['cv_percent'] for s in styles_list]
    axes[1, 0].bar(x_pos, cv_values, color='lightcoral', edgecolor='black', alpha=0.7)
    axes[1, 0].set_title('Metric Stability (CV% - Lower is Better)', fontweight='bold')
    axes[1, 0].set_xticks(x_pos)
    axes[1, 0].set_xticklabels(styles_list, rotation=45, ha='right')

    # 4. Metrics Heatmap
    metrics_matrix = np.array([[per_class_stats[s]['test']['f1']['mean'],
                                per_class_stats[s]['test']['precision']['mean'],
                                per_class_stats[s]['test']['recall']['mean']] for s in styles_list])
    sns.heatmap(metrics_matrix, annot=True, fmt='.3f', cmap='YlOrRd',
                xticklabels=['F1', 'Prec', 'Rec'], yticklabels=styles_list, ax=axes[1, 1])
    axes[1, 1].set_title('Metrics Heatmap', fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "per_class_analysis_summary.png"), dpi=300)
    plt.show()


In [ ]:
from google.colab import runtime
runtime.unassign()
